# DSA 504 — Homework 3
## Programmatic Data Access: Files and APIs

**Assigned:** Classes 11–12 (combined session, Tuesday, Oct 14)
**Due:** Before Class 14 (Tuesday, Oct 21)
**Worth:** 1 of 6 homework assignments (homework is 50% of your final grade, split evenly across all 6)

---

### Instructions

- Use `retail_sales_clean.csv` (the cleaned dataset from Class 7 onward) for Part 1.
- **Parts 2–4 require real internet access.** The APIs used (JSONPlaceholder and Open-Meteo) are free and require no sign-up or API key. If you're working somewhere without internet access, these cells will fail — that's expected, not a bug in your code.
- Fill in each `# TODO` code cell. Do not delete or reorder cells.
- A few cells ask a short written question — answer directly in the markdown cell provided.
- **AI use:** if you used an AI tool for any part of this assignment, briefly note where and how in the disclosure cell at the bottom, per the course AI-use policy in the syllabus.

### Submission — two required steps
1. Save this completed notebook inside a folder called `hw3/` in your `dsa504-coursework` GitHub Classroom repository, then commit and push:
```
git add hw3/
git commit -m "Complete HW3"
git push
```
2. Upload this completed `.ipynb` file directly to the HW3 assignment on Brightspace, and paste the direct link to your `hw3/` folder into the submission comments box.

**Do not email your GitHub link to the instructor — it will not be accepted as a submission.**

### Grading breakdown
| Section | Weight |
|---|---|
| Part 1 — File Formats | 20% |
| Part 2 — API Basics | 20% |
| Part 3 — Query Parameters & DataFrames | 20% |
| Part 4 — Real-World Weather API | 15% |
| Part 5 — Handling Credentials Safely | 15% |
| Part 6 — Written Reflection | 10% |


In [1]:
import pandas as pd
import numpy as np
import requests
import os

sales = pd.read_csv("retail_sales_clean.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales.head()


,date,store,units_sold,revenue,category,is_end_of_month
0,2025-04-22,Rome,25.0,402.50,toys,False
1,2025-04-12,Syracuse,38.0,3154.00,electronics,False
2,2025-12-21,Utica,42.0,1299.06,home goods,False
3,2025-02-11,Syracuse,60.0,5114.40,electronics,False
4,2025-04-10,Syracuse,10.0,147.10,toys,False


---
## Part 1 — File Formats (20%)

No internet access required for this part.


**1a.** Save `sales` to a JSON file called `sales_export.json`, using `orient="records"` and `date_format="iso"`. Read it back in with `pd.read_json()` and print its shape.


In [2]:
# TODO 1a
# Step 1: Export sales DataFrame to JSON
sales.to_json("sales_export.json", orient="records", date_format="iso")

# Step 2: Read the exported JSON file back into pandas
sales_json_imported = pd.read_json("sales_export.json")

# Step 3: Print the shape (number of rows and columns) to verify successful import
print("Shape of imported JSON file:", sales_json_imported.shape)

Shape of imported JSON file: (4014, 6)


**1b.** Save `sales` to an Excel file called `sales_export.xlsx`, without the index column. Read it back in and print its shape.


In [3]:
# TODO 1b
# Step 1: Export sales DataFrame to Excel without the index column
sales.to_excel("sales_export.xlsx", index=False)

# Step 2: Read the exported Excel file back into pandas
sales_excel_imported = pd.read_excel("sales_export.xlsx")

# Step 3: Print the shape to verify
print("Shape of imported Excel file:", sales_excel_imported.shape)


Shape of imported Excel file: (4014, 6)


**1c.** Save `sales` to both CSV and Parquet. Print the file size of each in KB, and state in a comment which one is smaller.


In [20]:
# TODO 1c
import os

# Step 1: Export the 'sales' DataFrame to CSV and Parquet formats
sales.to_csv("sales_export.csv", index=False)
sales.to_parquet("sales_export.parquet", index=False)

# Step 2: Get the file size of each in KB
csv_size = os.path.getsize("sales_export.csv") / 1024
parquet_size = os.path.getsize("sales_export.parquet") / 1024

print(f"CSV file size: {csv_size:.2f} KB")
print(f"Parquet file size: {parquet_size:.2f} KB")

# Step 3: State which one is smaller
# Parquet is smaller because it uses efficient columnar binary compression.


CSV file size: 182.28 KB
Parquet file size: 42.88 KB


---
## Part 2 — API Basics (20%)

**Requires internet access.**


**2a.** Send a GET request to `https://jsonplaceholder.typicode.com/users/5`. Print the status code, then print the user's name, email, and company name from the parsed JSON.


In [21]:
# TODO 2a
import requests

# Step 1: Send a GET request to the specified user endpoint
response = requests.get("https://jsonplaceholder.typicode.com/users/5")

# Step 2: Print the HTTP status code to verify success (200)
print("Status Code:", response.status_code)

# Step 3: Parse the response JSON and print user name, email, and company name
user_data = response.json()
print("Name:", user_data.get("name"))
print("Email:", user_data.get("email"))
print("Company:", user_data.get("company", {}).get("name"))


Status Code: 200
Name: Chelsey Dietrich
Email: Lucio_Hettinger@annie.ca
Company: Keebler LLC


**2b.** Using a try/except block with `.raise_for_status()`, request `https://jsonplaceholder.typicode.com/users/99999` (a user that doesn't exist) and print a clear message confirming whether the request succeeded or failed.


In [22]:
# TODO 2b
try:
    # Step 1: Request a non-existent user ID that triggers a 404 error
    response = requests.get("https://jsonplaceholder.typicode.com/users/99999")
    
    # Step 2: Raise an exception if the response code indicates an error
    response.raise_for_status()
    
    # Step 3: Print success message if no error occurred
    print("Request succeeded!")
except requests.exceptions.HTTPError as err:
    # Step 4: Handle failure gracefully and print confirmation message
    print(f"Request failed with error: {err}")


Request failed with error: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/users/99999


---
## Part 3 — Query Parameters & DataFrames (20%)

**Requires internet access.**


**3a.** Request all posts by `userId=3` from `https://jsonplaceholder.typicode.com/posts` using the `params=` argument. Print how many posts were returned.


In [23]:
# TODO 3a
# Step 1: Define query parameters as a dictionary
params = {"userId": 3}

# Step 2: Send GET request with params to JSONPlaceholder posts endpoint
response = requests.get("https://jsonplaceholder.typicode.com/posts", params=params)

# Step 3: Parse JSON and print how many posts were returned
posts = response.json()
print("Number of posts returned:", len(posts))


Number of posts returned: 10


**3b.** Convert the result from 3a into a DataFrame. Print its shape and its column names.


In [24]:
# TODO 3b
# Step 1: Convert the list of dictionaries from the API into a DataFrame
posts_df = pd.DataFrame(posts)

# Step 2: Print the shape of the DataFrame
print("DataFrame Shape:", posts_df.shape)

# Step 3: Print the column names
print("Column Names:", list(posts_df.columns))


DataFrame Shape: (10, 4)
Column Names: ['userId', 'id', 'title', 'body']


---
## Part 4 — Real-World Weather API (15%)

**Requires internet access.**


**4a.** Using Open-Meteo (`https://api.open-meteo.com/v1/forecast`), look up the current temperature and wind speed for a city of your choice (look up its latitude/longitude first — they don't need to be exact). Print a sentence like: `"The current temperature in [city] is [X]°C with wind speed [Y] km/h."`


In [25]:
# TODO 4a
# Step 1: Set up the API URL and query parameters for Syracuse, NY (approx. lat/lon)
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 43.0481,
    "longitude": -76.1474,
    "current": "temperature_2m,wind_speed_10m"
}

# Step 2: Send GET request to the Open-Meteo API
response = requests.get(url, params=params)
weather_data = response.json()

# Step 3: Extract current temperature and wind speed values from JSON response
temp = weather_data["current"]["temperature_2m"]
wind = weather_data["current"]["wind_speed_10m"]

# Step 4: Print the formatted sentence confirming the result
print(f'The current temperature in Syracuse is {temp}°C with wind speed {wind} km/h.')


The current temperature in Syracuse is 14.2°C with wind speed 16.3 km/h.


---
## Part 5 — Handling Credentials Safely (15%)

No internet access required — this part is about the *pattern*, not a live call.


**5a.** Write code that reads a value called `MY_API_KEY` from an environment variable using `os.environ.get()`. If it isn't set, print a clear message saying so instead of crashing. Do **not** hard-code any real key — this is testing the safe pattern, not a working credential.


In [26]:
# TODO 5a
# Step 1: Retrieve the API key from environment variables using os.environ.get()
api_key = os.environ.get("MY_API_KEY")

# Step 2: Check if the key exists and handle safely if missing instead of crashing
if api_key:
    print("API key successfully loaded.")
else:
    print("Warning: MY_API_KEY environment variable is not set.")

**5b. Written question:** Why is committing an API key directly in a script to GitHub a real security problem, even if you delete it in a later commit? Answer in 2–3 sentences.

*Your answer here:*


Committing an API key to GitHub makes it permanently visible in your repository's commit history, even if you delete or replace it in a later commit. Automated bots constantly scan public repositories for exposed credentials, allowing malicious actors to steal your key, abuse paid endpoints, or incur unexpected charges in your name.

---
## Part 6 — Written Reflection (10%)

Answer in 2–4 sentences, in this markdown cell:

**Question:** In Part 2b, you deliberately triggered a failed request. Why is it important to handle this kind of failure in code (with `.raise_for_status()` or a status-code check), rather than just assuming every request will succeed?

*Your answer here:*


It is crucial to handle request failures using .raise_for_status() or status-code checks because real-world APIs frequently encounter network issues, server downtime, or 404 errors. Assuming every request will succeed causes scripts to crash abruptly or silently process error text as valid JSON data, which breaks data pipelines and downstream analysis.

---
### AI Use Disclosure

*If you used an AI tool (Claude, ChatGPT, GitHub Copilot, etc.) for any part of this assignment, briefly state where and how below, and how you verified the result. If you did not use any AI tool, write "No AI tools were used."*

*Your answer here:*


I used an AI assistant (Gemini) for guidance with API request syntax, error handling patterns, and formatting step-by-step code comments. I independently verified all code by testing execution in my local Python environment.